
##                                  Credit Card Customer Segmentation Using KMeans Model

## Project Overview

Credit card companies manage thousands of customers with diverse spending patterns, credit usage, and repayment behaviors. However, treating all customers the same leads to ineffective marketing, poor risk assessment, and lost revenue opportunities. The problem is the lack of a data-driven approach to understand customer differences. This project aims to use the **K-Means clustering algorithm** on a **credit card dataset** to segment customers into meaningful groups based on their financial behavior. These segments will help the business personalize services, target marketing campaigns, improve customer retention, and identify high-value or high-risk customers.


### Loading the Python Packages

In [51]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import nbformat
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Import Data

In [3]:
df=pd.read_csv("/Users/HPPPPP/Downloads/segment/customer.csv")
df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


The Dataset summarizes the usage behavior of about 9000 active credit card holders during the last 6 months. The data is at a customer level with 18 behavioral variables.

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8636 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           8636 non-null   object 
 1   BALANCE                           8636 non-null   float64
 2   BALANCE_FREQUENCY                 8636 non-null   float64
 3   PURCHASES                         8636 non-null   float64
 4   ONEOFF_PURCHASES                  8636 non-null   float64
 5   INSTALLMENTS_PURCHASES            8636 non-null   float64
 6   CASH_ADVANCE                      8636 non-null   float64
 7   PURCHASES_FREQUENCY               8636 non-null   float64
 8   ONEOFF_PURCHASES_FREQUENCY        8636 non-null   float64
 9   PURCHASES_INSTALLMENTS_FREQUENCY  8636 non-null   float64
 10  CASH_ADVANCE_FREQUENCY            8636 non-null   float64
 11  CASH_ADVANCE_TRX                  8636 non-null   int64  
 12  PURCHASES_T

In [25]:
# Drop missing values
df.dropna(inplace=True)

## Explore

Since we are not given the feature to work on,the best way to choose the features for clustering is to determine which numerical features have the largest variance.

In [27]:
# Calculate the top ten variance
top_ten_var=df.drop(columns="CUST_ID").var().sort_values().tail(10)
top_ten_var

CASH_ADVANCE_TRX          4.778274e+01
PURCHASES_TRX             6.340560e+02
INSTALLMENTS_PURCHASES    8.413387e+05
ONEOFF_PURCHASES          2.836893e+06
BALANCE                   4.391419e+06
CASH_ADVANCE              4.500585e+06
PURCHASES                 4.696357e+06
MINIMUM_PAYMENTS          5.629071e+06
PAYMENTS                  8.466995e+06
CREDIT_LIMIT              1.339004e+07
dtype: float64

Use plotly express to create a horizontal bar chart of the `top_ten_var`. 

In [28]:
# Create horizontal bar chart of the top ten variance
fig=px.bar(x=top_ten_var,
           y=top_ten_var.index,
           title="High Variance Features"
           )
fig.update_layout(xaxis_title="Variance",yaxis_title="Feature")
fig.show();

We have to know if the top ten variance is highly skewed because those outliers can affect our measure of variance. Let's see if that's the case with one of the features from `top_five_var`.

In [29]:
# Create a boxplot of 'PURCHASES'
fig=px.box(x=df["PURCHASES"],title="Distribution of Purchases")
fig.update_layout(xaxis_title="Value [$]")
fig.show()

This dataset is rightly skewed with high outliers and we have to find solution to it. The best way to deal with this is to look at the **trimmed variance**, where we remove extreme values before calculating variance. We can do this using the `trimmed_variance` function from the `SciPy` library.

In [30]:
top_ten_trim_var=df.drop(columns="CUST_ID").apply(trimmed_var).sort_values().tail(10)
top_ten_trim_var

CASH_ADVANCE_TRX          6.464722e+00
PURCHASES_TRX             8.818169e+01
INSTALLMENTS_PURCHASES    8.823619e+04
MINIMUM_PAYMENTS          1.614683e+05
ONEOFF_PURCHASES          1.646010e+05
PURCHASES                 4.075150e+05
CASH_ADVANCE              6.288407e+05
PAYMENTS                  8.155695e+05
BALANCE                   1.155988e+06
CREDIT_LIMIT              5.367024e+06
dtype: float64

Use plotly express to create a horizontal bar chart of the `top_ten_trim_var`. 

In [31]:
# Create horizontal bar chart of the top ten trimmed variance
fig=px.bar(x=top_ten_trim_var,
           y=top_ten_trim_var.index,
           title="High Variance Features"
           )
fig.update_layout(xaxis_title="Variance",yaxis_title="Feature")
fig.show();

We observe that the xaxis scale have drop from $13M to $5.5M and some top 10 features have changed since we have trimmed the variance and then we will generate a list `high_var_cols` with the column names of the  five features with the highest trimmed variance.

In [32]:
high_var_cols=top_ten_trim_var.tail(5).index.to_list()
high_var_cols

['PURCHASES', 'CASH_ADVANCE', 'PAYMENTS', 'BALANCE', 'CREDIT_LIMIT']

## Split

Create the feature matrix `X`. It should contain the five columns in `high_var_cols`.

In [33]:
X=df[high_var_cols]
print("X type:", type(X))
print("X shape:", X.shape)
X.head()

X type: <class 'pandas.core.frame.DataFrame'>
X shape: (8636, 5)


,PURCHASES,CASH_ADVANCE,PAYMENTS,BALANCE,CREDIT_LIMIT
0,95.40,0.000000,201.802084,40.900749,1000.0
1,0.00,6442.945483,4103.032597,3202.467416,7000.0
2,773.17,0.000000,622.066742,2495.148862,7500.0
4,16.00,0.000000,678.334763,817.714335,1200.0
5,1333.28,0.000000,1400.057770,1809.828751,1800.0


## Build Model

#### K-Means Clustering

We use a `for` loop to build and train a K-Means model where `n_clusters` ranges from 2 to 12 (inclusive). My model will include a `StandardScaler`. Each time the model is trained, it will calculate the inertia and add it to the list `inertia_errors`, then calculate the silhouette score and add it to the list `silhouette_scores`.
 

In [34]:
n_clusters=range(2,13)
inertia_errors=[]
silhouette_scores=[]

# Add a for loop and calculate the inertia errors and silhouette scores
for k in n_clusters:
    # Build Kmeans model
    model=make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k,random_state=42)
    )
    # Fit Model
    model.fit(X)
    # Calculate the inertia by appending it to the inertia errors empty list
    inertia_errors.append(model.named_steps["kmeans"].inertia_)
    
    # Calculate the silhouette score
    silhouette_scores.append(silhouette_score(X,model.named_steps["kmeans"].labels_))


print("inertia_errors type:", type(inertia_errors))
print("inertia_errors len:", len(inertia_errors))
print("Inertia:", inertia_errors)
print()
print("silhouette_scores type:", type(silhouette_scores))
print("silhouette_scores len:", len(silhouette_scores))
print("Silhouette Scores:", silhouette_scores)

inertia_errors type: <class 'list'>
inertia_errors len: 11
Inertia: [30104.798815236267, 25119.445840719563, 21108.340276592317, 18205.93930217505, 16367.118974458182, 14300.221979785598, 13205.331940006334, 12470.965961924947, 11855.90320309283, 11289.397953060494, 10580.12379655929]

silhouette_scores type: <class 'list'>
silhouette_scores len: 11
Silhouette Scores: [np.float64(0.5465904221216482), np.float64(0.5223838343329856), np.float64(0.4404364003200464), np.float64(0.45959107776435093), np.float64(0.4000567264461099), np.float64(0.41886065720769183), np.float64(0.3894611833844458), np.float64(0.4017798277098505), np.float64(0.30647846715955174), np.float64(0.25354391174295565), np.float64(0.2802856130550617)]


We used plotly express to create a line plot that shows the values of `inertia_errors` as a function of `n_clusters`. Label my x-axis `"Number of Clusters"`, y-axis `"Inertia"`, and use the title `"K-Means Model: Inertia vs Number of Clusters"`

In [37]:
fig=px.line(x=n_clusters,
            y=inertia_errors,
            title="K-Means Model: Inertia vs Number of Clusters"
            )
fig.update_layout(xaxis_title="Number of Clusters",yaxis_title="Inertia")
fig.show()

We used plotly express to create a line plot that shows the values of `silhouette_scores` as a function of `n_clusters`. Label my x-axis `"Number of Clusters"`, y-axis `"Silhouette Score"`, and use the title `"K-Means Model: Silhouette Score vs Number of Clusters"`

In [39]:
fig=px.line(x=n_clusters,
            y=silhouette_scores,
            title="K-Means Model: Silhouette Scores vs Number of Clusters"
            )
fig.update_layout(xaxis_title="Number of Clusters",yaxis_title="Silhouette Scores")
fig.show()

 We can see that the best silhouette scores occur when there are 3 or 4 clusters. 
 
 Putting the information from this plot together with our inertia plot, it seems like the best setting for `n_clusters` will be 4. 


Then let rebuild our `final_model` with our `n_clusters` and with `StandardScaler`.

In [40]:
final_model=make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=4,random_state=42),\
)
final_model.fit(X)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=4, random_state=42))])

## Communicate

Extracted the labels of my `final_model` created during training and assign them to the variable `labels`.

In [43]:
labels=final_model.named_steps["kmeans"].labels_
print("labels type:", type(labels))
print("labels len:", len(labels))
print(labels[:5])

labels type: <class 'numpy.ndarray'>
labels len: 8636
[1 0 3 1 1]


Created a DataFrame `xgb` that contains the mean values of the features in `X` for each of the clusters in  `final_model`.

In [45]:
xgb=X.groupby(labels).mean()
xgb

,PURCHASES,CASH_ADVANCE,PAYMENTS,BALANCE,CREDIT_LIMIT
0,960.599101,5807.195620,5118.996678,6004.239828,9562.540253
1,515.708270,489.907381,917.224230,817.763476,2626.640448
2,22374.832703,1151.838978,23947.833531,5259.703654,15624.324324
3,2184.922805,708.343067,2732.947733,2245.840822,8143.884424


Used plotly express to create a side by side bar chart and label the axis.

In [46]:
# Create side-by-side bar chart of `xgb`
fig = px.bar(xgb,barmode="group",title="Mean of Credit Card by Cluster")
fig.update_layout(xaxis_title="Cluster",yaxis_title="Value [$]")
fig.show()

 ##### From this plot above we can be able to say that:

**Cluster 0:**

-Moderate PURCHASES and BALANCE.

-Low CASH_ADVANCE (means they don’t take much cash from cards).

-Mid-level CREDIT_LIMIT (~$10,000).

`Interpretation:`
 Average spenders — moderate usage, balanced payments, financially stable group.

**Cluster 1:**

-Everything is low (purchases, payments, cash advance, balance).

`Interpretation:`
 Low-activity users — people who rarely use their credit cards or are new cardholders. 

 **Cluster 2:**

-Very high PURCHASES and PAYMENTS — around $22K–$24K.

-High CREDIT_LIMIT (~$15K).

-Moderate BALANCE and low CASH_ADVANCE.

`Interpretation:`
High-value customers — they spend and repay large amounts; ideal, profitable segment for a bank.

**Cluster 3:**

-Moderate PURCHASES and PAYMENTS.

-Medium CREDIT_LIMIT (~$8K).

-Slightly higher BALANCE than cluster 1 but lower than cluster 0.

`Interpretation`:
Average but revolving users — they spend modestly but tend to carry some balance.

##### We will now visualize these customer segments in a 2D scatter plot using Principal Componenet Analysis (PCA) as cluster separation which would helpy me to see how distinct the clusters are.

In [47]:
# Instantiate transformer
pca = PCA(n_components=2,random_state=42)

# Transform `X`
X_t = pca.fit_transform(X)

# Put `X_t` into DataFrame
X_pca = pd.DataFrame(X_t,columns=["PC1","PC2"])

print("X_pca type:", type(X_pca))
print("X_pca shape:", X_pca.shape)
X_pca.head()

X_pca type: <class 'pandas.core.frame.DataFrame'>
X_pca shape: (8636, 2)


,PC1,PC2
0,-4319.249250,722.808147
1,4469.747404,-262.654556
2,1572.671349,-2776.792582
3,-3725.123423,754.518972
4,-2273.814669,1283.315205


In [50]:
# Create scatter plot of `PC2` vs `PC1`
fig =px.scatter(data_frame=X_pca,x="PC1",y="PC2",color=labels.astype(str),title="PCA Representation of Clusters")
fig.update_layout(xaxis_title="PC1",yaxis_title="PC2")
fig.show()

### Conclusion 

We can now conclude that:

-Cluster 2 is your best customers — possibly high-income, high-credit-limit individuals.

-Cluster 1 might need marketing re-engagement.

-Cluster 0 & 3 represent the middle class of users — good to keep but less profitable.